In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from IPython.display import display

pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

from pathlib import Path

def find_data_dir() -> Path:
    for base in (Path.cwd(), Path.cwd().parent):
        candidate = base / "Data"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find Data/ directory. Run the notebook from the repo root, Models/, or DataExploration/."
    )

DATA_DIR = find_data_dir()

try:
    file_path = DATA_DIR / "model_df.csv"
    model_df = pd.read_csv(file_path, na_values="?")
    print(f"Successfully loaded data from {file_path}")
    display(model_df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the file exists in the repo Data/ directory.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from IPython.display import display

pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

from pathlib import Path

def find_data_dir() -> Path:
    for base in (Path.cwd(), Path.cwd().parent):
        candidate = base / "Data"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find Data/ directory. Run the notebook from the repo root, Models/, or DataExploration/."
    )

DATA_DIR = find_data_dir()

try:
    file_path = DATA_DIR / "alphafold_variants.csv"
    alphafold_variants = pd.read_csv(file_path, na_values="?")
    print(f"Successfully loaded data from {file_path}")
    display(alphafold_variants.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the file exists in the repo Data/ directory.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
import re
import numpy as np

def extract_position(aa_change):
    """
    Extracts amino acid position from strings like:
    p.Thr148Ile -> 148
    p.Arg152His -> 152
    """
    if pd.isna(aa_change):
        return np.nan

    match = re.search(r"(\d+)", str(aa_change))

    if match:
        return int(match.group(1))

    return np.nan

alphafold_variants["protein_position"] = alphafold_variants["AA_change"].apply(extract_position)

alphafold_variants[[
    "Gene",
    "Entry",
    "AA_change",
    "protein_position",
    "Label",
    "num_predictors_failed"
]].head()

In [ ]:
import os
import requests
import pandas as pd

if "DATA_DIR" not in globals():
    DATA_DIR = find_data_dir()

STRUCTURES_DIR = DATA_DIR / "alphafold_structures"
os.makedirs(STRUCTURES_DIR, exist_ok=True)

def clean_uniprot_id(entry):
    if pd.isna(entry):
        return None

    entry = str(entry).strip()

    if "|" in entry:
        parts = entry.split("|")
        if len(parts) >= 2:
            entry = parts[1]

    # keep isoform if present for API check later, but base ID is useful too
    return entry


def get_alphafold_metadata(uniprot_id):
    """
    Queries AlphaFold API for a UniProt accession.
    Returns metadata if AlphaFold has a model.
    """
    uniprot_id = clean_uniprot_id(uniprot_id)

    possible_ids = [
        uniprot_id,
        uniprot_id.split("-")[0]
    ]

    for uid in possible_ids:
        url = f"https://alphafold.ebi.ac.uk/api/prediction/{uid}"

        try:
            r = requests.get(url, timeout=20)

            if r.status_code == 200:
                data = r.json()
                if len(data) > 0:
                    return data[0]

        except Exception as e:
            print(f"API error for {uid}: {e}")

    return None


def download_from_alphafold_api(uniprot_id, out_dir=None):
    if out_dir is None:
        out_dir = STRUCTURES_DIR
    """
    Downloads structure using the AlphaFold API-provided URL.
    Prefer CIF, fallback to PDB.
    """
    meta = get_alphafold_metadata(uniprot_id)

    if meta is None:
        print(f"No AlphaFold API result for {uniprot_id}")
        return None

    # API usually includes these fields
    download_urls = []

    if "cifUrl" in meta:
        download_urls.append(meta["cifUrl"])
    if "pdbUrl" in meta:
        download_urls.append(meta["pdbUrl"])

    for url in download_urls:
        filename = url.split("/")[-1]
        out_path = os.path.join(out_dir, filename)

        if os.path.exists(out_path):
            print(f"Already downloaded: {filename}")
            return out_path

        try:
            r = requests.get(url, timeout=30)

            if r.status_code == 200 and len(r.text) > 1000:
                with open(out_path, "w") as f:
                    f.write(r.text)

                print(f"Downloaded {uniprot_id}: {filename}")
                return out_path

        except Exception as e:
            print(f"Download error for {uniprot_id}: {e}")

    print(f"Metadata found but download failed for {uniprot_id}")
    return None


In [ ]:
unique_entries = alphafold_variants["Entry"].dropna().unique()

downloaded_files = {}

for entry in unique_entries:
    path = download_from_alphafold_api(entry)
    downloaded_files[entry] = path

downloaded_files

In [ ]:
import numpy as np
import pandas as pd

def parse_plddt_from_pdb(pdb_path):
    """
    Extract residue-level pLDDT from an AlphaFold PDB file.
    In AlphaFold PDB files, pLDDT is stored in the B-factor column.
    """
    residue_scores = {}

    with open(pdb_path, "r") as f:
        for line in f:
            if line.startswith("ATOM"):
                try:
                    residue_num = int(line[22:26].strip())
                    b_factor = float(line[60:66].strip())
                except:
                    continue

                residue_scores.setdefault(residue_num, []).append(b_factor)

    rows = []

    for residue_num, scores in residue_scores.items():
        rows.append({
            "protein_position": residue_num,
            "pLDDT": np.mean(scores)
        })

    return pd.DataFrame(rows)


def parse_plddt_from_cif(cif_path):
    """
    Extract residue-level pLDDT from AlphaFold mmCIF files.
    """
    rows = []
    in_atom_loop = False
    headers = []

    with open(cif_path, "r") as f:
        for line in f:
            line = line.strip()

            if line == "loop_":
                in_atom_loop = True
                headers = []
                continue

            if in_atom_loop and line.startswith("_atom_site."):
                headers.append(line)
                continue

            if in_atom_loop and headers and not line.startswith("_"):
                if line.startswith("#") or line == "":
                    in_atom_loop = False
                    continue

                parts = line.split()

                if len(parts) < len(headers):
                    continue

                header_map = {h: i for i, h in enumerate(headers)}

                try:
                    seq_idx = header_map.get("_atom_site.label_seq_id")
                    b_idx = header_map.get("_atom_site.B_iso_or_equiv")

                    if seq_idx is None or b_idx is None:
                        continue

                    residue_num = int(parts[seq_idx])
                    plddt = float(parts[b_idx])

                    rows.append({
                        "protein_position": residue_num,
                        "pLDDT_atom": plddt
                    })

                except:
                    continue

    if len(rows) == 0:
        return pd.DataFrame(columns=["protein_position", "pLDDT"])

    atom_df = pd.DataFrame(rows)

    residue_df = (
        atom_df
        .groupby("protein_position")["pLDDT_atom"]
        .mean()
        .reset_index()
        .rename(columns={"pLDDT_atom": "pLDDT"})
    )

    return residue_df


def parse_plddt_from_structure(path):
    if path is None:
        return pd.DataFrame(columns=["protein_position", "pLDDT"])

    if path.endswith(".pdb"):
        return parse_plddt_from_pdb(path)

    if path.endswith(".cif"):
        return parse_plddt_from_cif(path)

    return pd.DataFrame(columns=["protein_position", "pLDDT"])

In [ ]:
plddt_tables = []

for entry, structure_path in downloaded_files.items():
    if structure_path is None:
        continue

    plddt_df = parse_plddt_from_structure(structure_path)
    plddt_df["Entry"] = entry
    plddt_tables.append(plddt_df)

all_plddt = pd.concat(plddt_tables, ignore_index=True)

all_plddt.head()

In [ ]:
print("Number of residues with pLDDT:", len(all_plddt))
print(all_plddt.head())

In [ ]:
import re

def extract_position(aa_change):
    """
    Examples:
    p.Arg152His -> 152
    p.Gly12Asp -> 12
    """
    if pd.isna(aa_change):
        return np.nan

    match = re.search(r"(\d+)", str(aa_change))

    if match:
        return int(match.group(1))

    return np.nan


alphafold_variants["protein_position"] = alphafold_variants["AA_change"].apply(extract_position)

alphafold_variants[[
    "Gene",
    "Entry",
    "AA_change",
    "protein_position",
    "Label",
    "num_predictors_failed"
]].head()

In [ ]:
alphafold_variants = alphafold_variants.merge(
    all_plddt,
    on=["Entry", "protein_position"],
    how="left"
)

alphafold_variants[[
    "Gene",
    "Entry",
    "AA_change",
    "protein_position",
    "Label",
    "num_predictors_failed",
    "pLDDT"
]].head(20)

In [ ]:
def plddt_category(score):
    if pd.isna(score):
        return "Missing"
    elif score >= 90:
        return "Very high confidence"
    elif score >= 70:
        return "Confident"
    elif score >= 50:
        return "Low confidence"
    else:
        return "Very low confidence"


alphafold_variants["pLDDT_category"] = alphafold_variants["pLDDT"].apply(plddt_category)

alphafold_variants[[
    "Gene",
    "AA_change",
    "Label",
    "num_predictors_failed",
    "pLDDT",
    "pLDDT_category"
]].head()

In [ ]:
alphafold_variants["any_predictor_failed"] = alphafold_variants["num_predictors_failed"] > 0

alphafold_variants.groupby("any_predictor_failed")["pLDDT"].describe()

In [ ]:
import matplotlib.pyplot as plt

plot_df = alphafold_variants.dropna(subset=["pLDDT", "num_predictors_failed"]).copy()

groups = []
labels = []

for n in sorted(plot_df["num_predictors_failed"].unique()):
    group = plot_df[plot_df["num_predictors_failed"] == n]["pLDDT"]
    if len(group) > 0:
        groups.append(group)
        labels.append(f"{int(n)} failed")

plt.figure(figsize=(8, 5))

plt.boxplot(
    groups,
    labels=labels,
    patch_artist=True
)

plt.ylabel("AlphaFold pLDDT")
plt.xlabel("Number of Predictors Failed")
plt.title("AlphaFold Confidence by Number of Failed Predictors")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_df = alphafold_variants.dropna(subset=["pLDDT", "num_predictors_failed"]).copy()

categories = sorted(plot_df["num_predictors_failed"].unique())

plt.figure(figsize=(8, 5))

data = [
    plot_df[plot_df["num_predictors_failed"] == n]["pLDDT"]
    for n in categories
]

plt.boxplot(
    data,
    labels=[f"{int(n)} failed" for n in categories],
    patch_artist=True,
    showfliers=False
)

# Add jittered points
for i, n in enumerate(categories, start=1):
    y_vals = plot_df[plot_df["num_predictors_failed"] == n]["pLDDT"]
    x_vals = np.random.normal(i, 0.04, size=len(y_vals))

    plt.scatter(
        x_vals,
        y_vals,
        alpha=0.65,
        s=35
    )

plt.ylabel("AlphaFold pLDDT")
plt.xlabel("Number of Predictors Failed")
plt.title("AlphaFold Confidence by Predictor Failure Count")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
selected_gene_names = alphafold_variants["Gene"].dropna().unique()

alphafold_variants_all = model_df[
    model_df["Gene"].isin(selected_gene_names)
].copy()

In [ ]:
alphafold_variants_all["SIFT_pred_binary"] = (
    alphafold_variants_all["SIFT_score_clean"] < 0.05
).astype(int)

alphafold_variants_all["PolyPhen_pred_binary"] = (
    alphafold_variants_all["PolyPhen_score_clean"] > 0.909
).astype(int)

alphafold_variants_all["CADD_pred_binary"] = (
    alphafold_variants_all["CADD_phred_clean"] > 20
).astype(int)

alphafold_variants_all["REVEL_pred_binary"] = (
    alphafold_variants_all["REVEL_score_clean"] > 0.5
).astype(int)

alphafold_variants_all["SIFT_failed"] = (
    alphafold_variants_all["SIFT_pred_binary"] != alphafold_variants_all["y"]
)

alphafold_variants_all["PolyPhen_failed"] = (
    alphafold_variants_all["PolyPhen_pred_binary"] != alphafold_variants_all["y"]
)

alphafold_variants_all["CADD_failed"] = (
    alphafold_variants_all["CADD_pred_binary"] != alphafold_variants_all["y"]
)

alphafold_variants_all["REVEL_failed"] = (
    alphafold_variants_all["REVEL_pred_binary"] != alphafold_variants_all["y"]
)

failure_cols = [
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed"
]

alphafold_variants_all["num_predictors_failed"] = (
    alphafold_variants_all[failure_cols].sum(axis=1)
)

alphafold_variants_all["any_predictor_failed"] = (
    alphafold_variants_all["num_predictors_failed"] > 0
)

In [ ]:
alphafold_variants_all["protein_position"] = (
    alphafold_variants_all["AA_change"].apply(extract_position)
)

alphafold_variants_all = alphafold_variants_all.merge(
    all_plddt,
    on=["Entry", "protein_position"],
    how="left"
)

In [ ]:
no_failure = alphafold_variants_all[
    alphafold_variants_all["any_predictor_failed"] == False
]["pLDDT"].dropna()

failure = alphafold_variants_all[
    alphafold_variants_all["any_predictor_failed"] == True
]["pLDDT"].dropna()

print("No failure variants:", len(no_failure))
print("Failure variants:", len(failure))

plt.figure(figsize=(7, 5))

plt.boxplot(
    [no_failure, failure],
    labels=["No Predictor Failure", "At Least One Failure"],
    patch_artist=True
)

plt.ylabel("AlphaFold pLDDT")
plt.title("AlphaFold Confidence at Variant Sites")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
selected_genes_for_report = ["FGA", "HBD", "PROC", "GRM8"]

for gene in selected_genes_for_report:
    gene_df = alphafold_variants_all[
        alphafold_variants_all["Gene"] == gene
    ].dropna(subset=["protein_position", "pLDDT", "num_predictors_failed"])

    if len(gene_df) == 0:
        continue

    plt.figure(figsize=(9, 4))

    sizes = 50 + gene_df["num_predictors_failed"] * 45

    plt.scatter(
        gene_df["protein_position"],
        gene_df["pLDDT"],
        s=sizes,
        alpha=0.75,
        edgecolor="black",
        linewidth=0.5
    )

    plt.axhline(90, linestyle="--", linewidth=1)
    plt.axhline(70, linestyle="--", linewidth=1)
    plt.axhline(50, linestyle="--", linewidth=1)

    plt.text(gene_df["protein_position"].min(), 91, "Very high confidence", fontsize=9)
    plt.text(gene_df["protein_position"].min(), 71, "Confident", fontsize=9)
    plt.text(gene_df["protein_position"].min(), 51, "Low confidence", fontsize=9)

    plt.xlabel("Protein Position")
    plt.ylabel("AlphaFold pLDDT")
    plt.title(f"{gene}: Predictor Failures Across Protein Sequence")
    plt.ylim(20, 102)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
interesting_variants = (
    alphafold_variants
    .sort_values(
        ["num_predictors_failed", "pLDDT"],
        ascending=[False, True]
    )
)

interesting_variants[[
    "Gene",
    "Entry",
    "AA_change",
    "protein_position",
    "Label",
    "Disease",
    "num_predictors_failed",
    "pLDDT",
    "pLDDT_category",
    "SIFT_failed",
    "PolyPhen_failed",
    "CADD_failed",
    "REVEL_failed"
]].head(30)

In [ ]:
import pandas as pd
import numpy as np

# Use the full AlphaFold table with both failed and non-failed variants
plot_df = alphafold_variants_all.dropna(subset=["pLDDT", "num_predictors_failed"]).copy()

gene_summary = (
    plot_df
    .groupby("Gene")
    .agg(
        total_variants=("Gene", "count"),
        failed_variants=("any_predictor_failed", "sum"),
        mean_predictors_failed=("num_predictors_failed", "mean"),
        median_predictors_failed=("num_predictors_failed", "median"),
        max_predictors_failed=("num_predictors_failed", "max"),
        mean_pLDDT=("pLDDT", "mean"),
        median_pLDDT=("pLDDT", "median"),
        min_pLDDT=("pLDDT", "min"),
        low_confidence_variants=("pLDDT", lambda x: (x < 70).sum()),
        very_low_confidence_variants=("pLDDT", lambda x: (x < 50).sum())
    )
    .reset_index()
)

gene_summary["failure_rate"] = (
    gene_summary["failed_variants"] / gene_summary["total_variants"]
)

gene_summary["low_confidence_rate"] = (
    gene_summary["low_confidence_variants"] / gene_summary["total_variants"]
)

gene_summary["very_low_confidence_rate"] = (
    gene_summary["very_low_confidence_variants"] / gene_summary["total_variants"]
)

gene_summary = gene_summary.sort_values(
    ["failure_rate", "mean_predictors_failed", "low_confidence_rate"],
    ascending=False
)

gene_summary

In [ ]:
report_gene_summary = gene_summary[[
    "Gene",
    "total_variants",
    "failed_variants",
    "failure_rate",
    "mean_predictors_failed",
    "mean_pLDDT",
    "median_pLDDT",
    "low_confidence_rate",
    "very_low_confidence_rate"
]].copy()

report_gene_summary["failure_rate"] = report_gene_summary["failure_rate"].round(3)
report_gene_summary["mean_predictors_failed"] = report_gene_summary["mean_predictors_failed"].round(2)
report_gene_summary["mean_pLDDT"] = report_gene_summary["mean_pLDDT"].round(2)
report_gene_summary["median_pLDDT"] = report_gene_summary["median_pLDDT"].round(2)
report_gene_summary["low_confidence_rate"] = report_gene_summary["low_confidence_rate"].round(3)
report_gene_summary["very_low_confidence_rate"] = report_gene_summary["very_low_confidence_rate"].round(3)

report_gene_summary

In [ ]:
import matplotlib.pyplot as plt

bubble_df = gene_summary.copy()

plt.figure(figsize=(10, 6))

plt.scatter(
    bubble_df["mean_pLDDT"],
    bubble_df["failure_rate"],
    s=bubble_df["total_variants"] * 35,
    alpha=0.65,
    edgecolor="black",
    linewidth=0.7
)

for _, row in bubble_df.iterrows():
    plt.text(
        row["mean_pLDDT"] + 0.4,
        row["failure_rate"] + 0.005,
        row["Gene"],
        fontsize=9
    )

plt.axvline(70, linestyle="--", linewidth=1, label="Low confidence cutoff")
plt.axvline(90, linestyle="-", linewidth=1, label="Very high confidence cutoff")

plt.xlabel("Mean AlphaFold pLDDT")
plt.ylabel("Predictor Failure Rate")
plt.title("Gene-Level Predictor Failure Rate vs AlphaFold Confidence")
plt.ylim(-0.05, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
top_gene_plot = gene_summary.head(15).copy()

plt.figure(figsize=(10, 6))

plt.barh(
    top_gene_plot["Gene"],
    top_gene_plot["failure_rate"]
)

plt.xlabel("Predictor Failure Rate")
plt.ylabel("Gene")
plt.title("Top Genes by Predictor Failure Rate")
plt.xlim(0, 1)
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
top_avg_fail = gene_summary.sort_values(
    "mean_predictors_failed",
    ascending=False
).head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top_avg_fail["Gene"],
    top_avg_fail["mean_predictors_failed"]
)

plt.xlabel("Average Number of Predictors Failed")
plt.ylabel("Gene")
plt.title("Genes with Highest Average Predictor Failure Count")
plt.xlim(0, 4)
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
def plddt_category_short(score):
    if pd.isna(score):
        return "Missing"
    elif score >= 90:
        return "Very high"
    elif score >= 70:
        return "Confident"
    elif score >= 50:
        return "Low"
    else:
        return "Very low"

plot_df["pLDDT_category_short"] = plot_df["pLDDT"].apply(plddt_category_short)

category_order = ["Very high", "Confident", "Low", "Very low"]

confidence_by_gene = (
    plot_df
    .groupby(["Gene", "pLDDT_category_short"])
    .size()
    .reset_index(name="count")
)

confidence_pivot = confidence_by_gene.pivot(
    index="Gene",
    columns="pLDDT_category_short",
    values="count"
).fillna(0)

for col in category_order:
    if col not in confidence_pivot.columns:
        confidence_pivot[col] = 0

confidence_pivot = confidence_pivot[category_order]

# Sort genes by failure rate from previous summary
gene_order = gene_summary["Gene"].tolist()
confidence_pivot = confidence_pivot.loc[
    [g for g in gene_order if g in confidence_pivot.index]
]

confidence_pivot.plot(
    kind="barh",
    stacked=True,
    figsize=(10, 6)
)

plt.xlabel("Number of Variants")
plt.ylabel("Gene")
plt.title("AlphaFold Confidence Categories by Gene")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
failed_only = plot_df[plot_df["any_predictor_failed"] == True].copy()

failed_only["pLDDT_category_short"] = failed_only["pLDDT"].apply(plddt_category_short)

failed_confidence_by_gene = (
    failed_only
    .groupby(["Gene", "pLDDT_category_short"])
    .size()
    .reset_index(name="count")
)

failed_confidence_pivot = failed_confidence_by_gene.pivot(
    index="Gene",
    columns="pLDDT_category_short",
    values="count"
).fillna(0)

for col in category_order:
    if col not in failed_confidence_pivot.columns:
        failed_confidence_pivot[col] = 0

failed_confidence_pivot = failed_confidence_pivot[category_order]

gene_order = gene_summary["Gene"].tolist()
failed_confidence_pivot = failed_confidence_pivot.loc[
    [g for g in gene_order if g in failed_confidence_pivot.index]
]

failed_confidence_pivot.plot(
    kind="barh",
    stacked=True,
    figsize=(10, 6)
)

plt.xlabel("Number of Failed Variants")
plt.ylabel("Gene")
plt.title("AlphaFold Confidence Categories Among Failed Variants")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu

failed_plddt = plot_df[
    plot_df["any_predictor_failed"] == True
]["pLDDT"].dropna()

nonfailed_plddt = plot_df[
    plot_df["any_predictor_failed"] == False
]["pLDDT"].dropna()

stat, p_value = mannwhitneyu(
    failed_plddt,
    nonfailed_plddt,
    alternative="two-sided"
)

print("Mann-Whitney U test")
print("Failed variants n:", len(failed_plddt))
print("Non-failed variants n:", len(nonfailed_plddt))
print("U statistic:", stat)
print("p-value:", p_value)
print("Failed median pLDDT:", failed_plddt.median())
print("Non-failed median pLDDT:", nonfailed_plddt.median())

In [ ]:
median_diff = failed_plddt.median() - nonfailed_plddt.median()
mean_diff = failed_plddt.mean() - nonfailed_plddt.mean()

print("Median pLDDT difference, failed - nonfailed:", median_diff)
print("Mean pLDDT difference, failed - nonfailed:", mean_diff)

In [ ]:
from scipy.stats import spearmanr

corr_df = plot_df.dropna(subset=["pLDDT", "num_predictors_failed"])

rho, p_value = spearmanr(
    corr_df["pLDDT"],
    corr_df["num_predictors_failed"]
)

print("Spearman correlation")
print("rho:", rho)
print("p-value:", p_value)

In [ ]:
import statsmodels.api as sm

logit_df = plot_df.dropna(subset=["pLDDT", "any_predictor_failed"]).copy()

logit_df["failure_binary"] = logit_df["any_predictor_failed"].astype(int)

X = logit_df[["pLDDT"]]
X = sm.add_constant(X)

y = logit_df["failure_binary"]

logit_model = sm.Logit(y, X).fit()

print(logit_model.summary())

In [ ]:
params = logit_model.params
conf = logit_model.conf_int()

odds_ratios = np.exp(params)
conf_odds = np.exp(conf)

odds_table = pd.DataFrame({
    "Odds Ratio": odds_ratios,
    "CI Lower": conf_odds[0],
    "CI Upper": conf_odds[1],
    "p-value": logit_model.pvalues
})

odds_table

In [ ]:
import statsmodels.formula.api as smf

logit_df = plot_df.dropna(subset=["pLDDT", "any_predictor_failed", "Gene"]).copy()
logit_df["failure_binary"] = logit_df["any_predictor_failed"].astype(int)

gene_logit = smf.logit(
    formula="failure_binary ~ pLDDT + C(Gene)",
    data=logit_df
).fit()

print(gene_logit.summary())

In [ ]:
mw_stat, mw_p = mannwhitneyu(
    failed_plddt,
    nonfailed_plddt,
    alternative="two-sided"
)

rho, spearman_p = spearmanr(
    corr_df["pLDDT"],
    corr_df["num_predictors_failed"]
)

stat_summary = pd.DataFrame({
    "Analysis": [
        "Failed median pLDDT",
        "Non-failed median pLDDT",
        "Median difference",
        "Mann-Whitney p-value",
        "Spearman rho",
        "Spearman p-value"
    ],
    "Value": [
        failed_plddt.median(),
        nonfailed_plddt.median(),
        failed_plddt.median() - nonfailed_plddt.median(),
        mw_p,
        rho,
        spearman_p
    ]
})

stat_summary

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

stat_df = plot_df.dropna(subset=["pLDDT", "any_predictor_failed"]).copy()

stat_df["low_confidence"] = stat_df["pLDDT"] < 70

contingency = pd.crosstab(
    stat_df["low_confidence"],
    stat_df["any_predictor_failed"]
)

print(contingency)

odds_ratio, fisher_p = fisher_exact(contingency)

print("Fisher exact test")
print("Odds ratio:", odds_ratio)
print("p-value:", fisher_p)

In [ ]:
stat_df["very_low_confidence"] = stat_df["pLDDT"] < 50

contingency_very_low = pd.crosstab(
    stat_df["very_low_confidence"],
    stat_df["any_predictor_failed"]
)

print(contingency_very_low)

odds_ratio_vl, fisher_p_vl = fisher_exact(contingency_very_low)

print("Very-low confidence Fisher exact test")
print("Odds ratio:", odds_ratio_vl)
print("p-value:", fisher_p_vl)